#Order Trend Analysis

This notebook analyzes how order volume changes over time.

Focus areas:
- Yearly order growth
- Monthly seasonality
- Order timing by part of day
- Month-on-month order volume by customer state

In [0]:
/*
Yearly Order Trend

Purpose:
    Analyze the number of orders placed each year to identify whether
    e-commerce order volume is growing over time.
*/

WITH yearly_orders AS (
    SELECT
        YEAR(order_purchase_timestamp) AS order_year,
        COUNT(DISTINCT order_id) AS total_orders
    FROM ecommerce_analysis.orders
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY YEAR(order_purchase_timestamp)
)

SELECT
    order_year,
    total_orders,
    LAG(total_orders) OVER (ORDER BY order_year) AS previous_year_orders,
    total_orders - LAG(total_orders) OVER (ORDER BY order_year) AS order_growth,
    ROUND(
        (total_orders - LAG(total_orders) OVER (ORDER BY order_year)) * 100.0
        / LAG(total_orders) OVER (ORDER BY order_year),
        2
    ) AS growth_percentage
FROM yearly_orders
ORDER BY order_year;

In [0]:
/*
Monthly Order Seasonality

Purpose:
    Aggregate orders by month of year to check whether certain months
    consistently receive more orders.
*/
SELECT
    MONTH(order_purchase_timestamp) AS month_order,
    COUNT(DISTINCT order_id) AS num_orders
FROM ecommerce_analysis.orders
GROUP BY MONTH(order_purchase_timestamp)
ORDER BY month_order;


In [0]:
/*
Order Time-of-Day Segmentation

Purpose:
    Segment order purchase timestamps into Dawn, Morning, Afternoon,
    and Night to understand when customers usually place orders.
*/
WITH order_time_segments AS (
    SELECT
        order_id,
        CASE
            WHEN HOUR(order_purchase_timestamp) BETWEEN 0 AND 6 THEN 'Dawn'
            WHEN HOUR(order_purchase_timestamp) BETWEEN 7 AND 12 THEN 'Morning'
            WHEN HOUR(order_purchase_timestamp) BETWEEN 13 AND 18 THEN 'Afternoon'
            WHEN HOUR(order_purchase_timestamp) BETWEEN 19 AND 23 THEN 'Night'
        END AS time_segment
    FROM ecommerce_analysis.orders
    WHERE order_purchase_timestamp IS NOT NULL
)

SELECT
    time_segment,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(
        COUNT(DISTINCT order_id) * 100.0 / SUM(COUNT(DISTINCT order_id)) OVER (),
        2
    ) AS order_percentage
FROM order_time_segments
GROUP BY time_segment
ORDER BY total_orders DESC;

In [0]:
/*
Monthly Orders by State

Purpose:
    Calculate month-on-month order volume for each customer state.
    This helps analyze regional order evolution over time.
*/

WITH monthly_state_orders AS (
    SELECT
        c.customer_state AS state,
        DATE_TRUNC('MONTH', o.order_purchase_timestamp) AS order_month,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM ecommerce_analysis.orders o
    LEFT JOIN ecommerce_analysis.customer c
        ON o.customer_id = c.customer_id
    WHERE o.order_purchase_timestamp IS NOT NULL
    GROUP BY
        c.customer_state,
        DATE_TRUNC('MONTH', o.order_purchase_timestamp)
)

SELECT
    COALESCE(state, 'Unknown') AS state,
    order_month,
    total_orders,
    LAG(total_orders) OVER (
        PARTITION BY state
        ORDER BY order_month
    ) AS previous_month_orders,
    total_orders - LAG(total_orders) OVER (
        PARTITION BY state
        ORDER BY order_month
    ) AS month_over_month_change
FROM monthly_state_orders
ORDER BY state, order_month;